# Cell Segmentation Inference Workflow

This notebook demonstrates how to perform cell segmentation on microscopy images using the Cellpose model. It covers single image inference, batch processing, configuration-based workflows, and custom model integration, providing a reproducible pipeline for analyzing cell morphology and generating segmentation masks.

In [ ]:
# Import essential modules for cell segmentation with Cellpose
from src.inference.cellpose_predictor import CellposePredictor
from src.inference.output_manager import OutputManager
from src.inference.inference_pipeline import InferencePipeline

# For file path operations
from pathlib import Path


In [1]:

# Test dataset path - contains brightfield images to segment
input_dir = "../data/Plate 2426_preprocessed_2D/test"

# Output directory for segmentation masks and visualization
output_dir = "../data/Plate 2426_preprocessed_2D/inference"

## Single Image Inference

This section demonstrates how to use Cellpose for segmenting a single microscopy image. 
Single image inference is useful for:

- Testing the pipeline on individual samples before batch processing
- Debugging segmentation parameters on specific examples
- Quickly visualizing results for model evaluation

The basic workflow is:
1. Initialize the Cellpose predictor with model settings
2. Configure output handling
3. Set up the pipeline
4. Run inference on a single image
5. Visualize and evaluate the results

In [ ]:
"""Example of running inference on a single brightfield microscopy image file."""

print("\n=== Single File Inference Example ===")

# Initialize predictor
predictor = CellposePredictor(model_type="cyto3", gpu=True)

# Set up output : results/cyto3_single/example/
output_manager = OutputManager(
    base_output_dir="results",
    model_name="cyto3_single",
    dataset_name="example"
)

# Create pipeline
pipeline = InferencePipeline(predictor, output_manager)

# Find a test file
test_dir = Path(input_dir)
test_files = list(test_dir.glob("*_BF.tif")) if test_dir.exists() else []

# Get the first test file if available
test_file = test_files[0] if test_files else None

# Check that we found at least one test image
assert test_file is not None, f"No test files found in {test_dir}"

print(f"Processing single file: {test_file}")

# Run inference on the single test file
# This returns a dictionary with results including mask, status, etc.
result = pipeline.run_inference_single(test_file)

print(f"Result:")
print(f"  - Status: {result['status']}")
print(f"  - Cells detected: {result.get('num_cells', 0)}")
print(f"  - Image shape: {result.get('image_shape', 'Unknown')}")



In [ ]:
import napari

# Visualize the result using napari
if result['status'] == 'success' and 'image' in result and 'mask' in result:
    # Create a napari viewer instance
    viewer = napari.Viewer()
    
    # Add the original image
    viewer.add_image(result['image'], name='Original Image')
    
    # Add the segmentation mask
    viewer.add_labels(result['mask'], name='Cell Segmentation')
    
    # If flow data is available, can add that as well
    if 'flows' in result:
        viewer.add_image(result['flows'][0], name='Cellpose Flows', colormap='hsv')
    
    print("Visualization opened in napari viewer")
else:
    print("Unable to visualize results - missing image or mask data")

## Batch Inference Processing

This section demonstrates how to run Cellpose segmentation on a directory of images.
Batch inference is essential for:

- Processing large experimental datasets efficiently
- Ensuring consistent segmentation across all samples
- Generating comprehensive results for statistical analysis

The batch processing workflow follows these steps:

1. **Configure the Cellpose predictor** - Set model type and parameters for optimal segmentation
2. **Set up output management** - Define where and how to save results 
3. **Create and validate the pipeline** - Connect components and verify setup
4. **Run inference on all images** - Process all matching files in the input directory
5. **Analyze aggregate results** - Summarize cell counts and processing statistics

Batch processing automatically handles file discovery, parallel processing (when available), and organized output storage.

In [ ]:
# 1. Initialize the predictor
predictor = CellposePredictor(
    model_type="cyto3",
    gpu=True,
    flow_threshold=0.4,  # Flow threshold controls detection sensitivity
    min_size=30          # Min size filters out small artifacts/noise
)

# 2. Set up output management
output_manager = OutputManager(
    base_output_dir="results",
    model_name="cyto3",
    dataset_name="test"
)

# 3. Create pipeline
pipeline = InferencePipeline(predictor, output_manager)

# 4. Validate setup
validation = pipeline.validate_setup()
if not validation['overall']:
    raise RuntimeError(f"Pipeline setup validation failed: {validation}")

# 5. Run inference on test directory
test_dir = "data/sample_plates_split/test"

assert Path(test_dir).exists(), f"Test directory not found: {test_dir}"

print(f"Running inference on: {test_dir}")

# Execute batch inference on all brightfield images in directory
results = pipeline.run_inference(
    input_dir=test_dir,
    file_pattern="*_BF.tif",  # Pattern to match brightfield images
    save_overlays=True,       # Generate and save overlay visualizations
    save_metadata=True        # Save analysis parameters and results
)

print(f"\nResults:")
print(f"  - Files processed: {len(results['processed_files'])}")
print(f"  - Total cells detected: {results['total_cells']}")
print(f"  - Output directory: {output_manager.output_dir}")



In [ ]:
from matplotlib import pyplot as plt

# 7. Optional: Calculate basic statistics on cell counts
if len(results['processed_files']) > 0 and 'cell_counts' in results:
    cell_counts = results['cell_counts']
    print(f"  - Average cells per image: {sum(cell_counts) / len(cell_counts):.2f}")
    print(f"  - Min cells: {min(cell_counts)}, Max cells: {max(cell_counts)}")
    
    # Plot distribution of cell counts
    plt.figure(figsize=(10, 5))
    plt.hist(cell_counts, bins=10)
    plt.title("Distribution of Detected Cells per Image")
    plt.xlabel("Number of Cells")
    plt.ylabel("Count")
    plt.show()

## Configuration-Based Inference

This section demonstrates how to run Cellpose inference using external YAML configuration files.
Using configuration files provides several advantages:

- **Reproducibility**: Record exact parameters for each experiment
- **Flexibility**: Easily switch between parameter sets without code changes
- **Documentation**: Configuration files serve as documentation for processing settings
- **Batch processing**: Run multiple configurations across datasets

The configuration file (`config/inference_config.yaml`) typically contains:
- Model parameters (model type, weights, thresholds)
- Processing settings (batch size, use GPU)
- Input/output specifications
- Visualization options

This approach allows for standardized processing across multiple experiments and datasets.

In [ ]:
# Specify the path to configuration file
config_path = "config/inference_config.yaml"

# Ensure the configuration file exists
assert Path(config_path).exists(), f"Config file not found: {config_path}"

pipeline = InferencePipeline.from_config(
    config_path=config_path,
    model_name="cyto3", 
    output_dir="results",
    dataset_name="test_example"
)

# Get test directory from config
test_dir = Path(input_dir)

# Verify the test directory exists
assert Path(test_dir).exists(), f"Test directory not found: {test_dir}"

print(f"Running inference with config on: {test_dir}")

results = pipeline.run_inference(
    input_dir=test_dir,
    file_pattern="*_BF.tif"
)

# Print summary statistics from batch processing
print(f"\nResults:")
print(f"  - Files processed: {len(results['processed_files'])}")
print(f"  - Total cells detected: {results['total_cells']}")


## Using Custom Trained Models

This section demonstrates how to use a custom Cellpose model for inference.

The process for using a custom model involves:
1. Training the model (separate process)
2. Saving the model weights
3. Loading the model in the predictor
4. Running inference with the custom model


In [ ]:
# Check if custom model exists
custom_model_path = "models/custom_cellpose_model"

assert Path(custom_model_path).exists(), f"Custom model not found at: {custom_model_path}"

# Initialize with custom model
predictor = CellposePredictor(model_type="cyto3", gpu=True)

# Load the custom model weights
predictor.load_model(custom_model_path)
    
# Configure output management
output_manager = OutputManager(
    base_output_dir="results",
    model_name="custom_model",
    dataset_name="test"
)
    
pipeline = InferencePipeline(predictor, output_manager)

print(f"Loaded custom model from: {custom_model_path}")
print(f"Model info: {pipeline.get_model_info()}")
    
